In [1]:
! pip install pandas python-dotenv openpyxl xlrd>=2.0.1 -q


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import json
import pandas as pd
from dotenv import load_dotenv
from pathlib import Path

load_dotenv()

BASE_PATH = "../1 - scrapping/"
THIS_FOLDER = "../2 - data_handling/"
MANUAL_FILTERING_PATH = "../3 - manual_filtering/"


## ChatGPT Agent

In [ ]:
import json
from typing import List
from openai import OpenAI

client = OpenAI()

def evaluate_paper(
    title: str,
    abstract: str,
    inclusion_criteria: List[str],
    exclusion_criteria: List[str],
    model: str = "gpt-4o-mini"
) -> List[bool]:
    """
    Evaluates a publication title + abstract against inclusion and exclusion criteria.

    Returns:
        A boolean list:
        [IC1, IC2, ..., ICn, EC1, EC2, ..., ECm]
    """

    system_prompt = """
You are assisting in a systematic literature review.

You will receive:
- A publication title
- A publication abstract
- A list of inclusion criteria
- A list of exclusion criteria

For EACH inclusion criterion:
Return true if the paper satisfies it.
Return false if it does not.

For EACH exclusion criterion:
Return true if the paper meets the exclusion condition.
Return false if it does not.

You must respond ONLY with valid JSON in this exact format:

{
  "results": {"IC I": true, "IC II": false, ..., "EC I": false, "EC II": true, ...}
}

"""

    user_prompt = f"""
TITLE:
{title}

ABSTRACT:
{abstract}

INCLUSION CRITERIA:
{json.dumps(inclusion_criteria, indent=2)}

EXCLUSION CRITERIA:
{json.dumps(exclusion_criteria, indent=2)}
"""

    response = client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )

    content = response.choices[0].message.content.strip()

    try:
        parsed = json.loads(content)
    except json.JSONDecodeError:
        raise ValueError(f"Model did not return valid JSON:\n{content}")

    if "results" not in parsed:
        raise ValueError("Missing 'results' key in response.")

    results = parsed["results"]

    expected_length = len(inclusion_criteria) + len(exclusion_criteria)

    if (
        not isinstance(results, dict)
        or len(results.keys()) != expected_length
        or not all(isinstance(x, bool) for x in results.values())
    ):
        raise ValueError(
            f"Invalid response format. Expected {expected_length} booleans, got: {results}"
        )

    return results

In [86]:

inclusion = [
    "IC I (Methodological Scope): The research must explicitly apply, evaluate, or propose textual feature extraction or representation mechanisms.",
    "IC II (Model Architecture): The formulated textual representations must be systematically operationalized as independent variables (covariates) within a predictive regression framework.",
]

exclusion = [
    "EC I (Application Deficit): Investigations lacking a direct, empirical implementation within regression analysis. ",
    "EC II (Theoretical Isolation): Purely theoretical NLP discourse devoid of applicability to textual representation for supervised machine learning downstream tasks.",
    "EC III (Thematic Misalignment):  Literature fundamentally omitting discussion of either NLP methodologies or regression techniques.",
    "EC V (Target Variable Formulation): Target variable not in $y \\in \\mathbb{R}$), which means categorical or discrete outputs."
]


## Importing

In [3]:

with open(f'{BASE_PATH}/search_strings.json', 'r') as f:
    search_string_json = json.load(f)

categories = [category.get("category") for category in search_string_json]
categories

['General',
 'Classic Methods',
 'Dense Embeddings',
 'Contextual Embeddings',
 'Hybrid Methods',
 'Comparison of methods',
 'Literature Reviews',
 'Applied cases']

In [6]:

def read_list_of_csvs(file_list) -> pd.DataFrame:
    df_list = []
    for file in file_list:
        df = pd.read_csv(file)
        df['filename'] = file
        df_list.append(df)
    combined_df = pd.concat(df_list, ignore_index=True)
    return combined_df

def read_list_of_excels(file_list) -> pd.DataFrame:
    df_list = []
    for file in file_list:
        df = pd.read_excel(file)
        df['filename'] = file
        df_list.append(df)
    combined_df = pd.concat(df_list, ignore_index=True)
    return combined_df


In [8]:
# Google Scholar
google_scholar_files = [f"{BASE_PATH}/result/google_scholar_search_results.csv"]
df_google_scholar = read_list_of_csvs(google_scholar_files)
df_google_scholar.head()


,str_search_string,nr_page,nr_order,str_article,nr_year,nr_citations,str_original_Link,str_pdf_link,category,filename
0,"""text regression"" AND (""feature extraction"" OR...",1,1,"Text regressionanalysis: A review, empirical, ...",2024.0,3,https://ieeexplore.ieee.org/abstract/document/...,https://ieeexplore.ieee.org/iel8/6287639/10380...,General,../1 - scrapping//result/google_scholar_search...
1,"""text regression"" AND (""feature extraction"" OR...",1,2,Multi-representation approach totext regressio...,2015.0,8,https://ieeexplore.ieee.org/abstract/document/...,https://www.fruct.org/files/publications/volum...,General,../1 - scrapping//result/google_scholar_search...
2,"""text regression"" AND (""feature extraction"" OR...",1,3,[HTML]TextRegress: A Python package for advanc...,2025.0,0,https://www.sciencedirect.com/science/article/...,https://www.sciencedirect.com/science/article/...,General,../1 - scrapping//result/google_scholar_search...
3,"""text regression"" AND (""feature extraction"" OR...",1,4,Gaussian processes fortext regression,2017.0,5,https://etheses.whiterose.ac.uk/id/eprint/17619/,https://etheses.whiterose.ac.uk/id/eprint/1761...,General,../1 - scrapping//result/google_scholar_search...
4,"""text regression"" AND (""feature extraction"" OR...",1,5,Analyzing online review helpfulness using a re...,2012.0,52,https://dl.acm.org/doi/abs/10.1145/2229156.222...,https://dl.acm.org/doi/pdf/10.1145/2229156.222...,General,../1 - scrapping//result/google_scholar_search...


In [7]:
# Springer International Journal of Data Science and Analytics (IJDSR)
springer_ijdsr_files = [
    f"{BASE_PATH}/result/springer_ijdsr_search_results_{category}.csv"
    for category in categories
]

df_springer_ijdsr = read_list_of_csvs(springer_ijdsr_files)
df_springer_ijdsr.head()

,Item Title,Publication Title,Book Series Title,Journal Volume,Journal Issue,Item DOI,Authors,Publication Year,URL,Content Type,filename
0,Large-scale simulation study of active learnin...,International Journal of Data Science and Anal...,NaN,NaN,NaN,10.1007/s41060-025-00777-0,Jelle Jasper TeijemaJonathan de BruinAyoub Bag...,2025,https://link.springer.com/article/10.1007/s410...,Article,../1 - scrapping//result/springer_ijdsr_search...
1,A survey of sentiment analysis methods based o...,International Journal of Data Science and Anal...,NaN,NaN,NaN,10.1007/s41060-025-00714-1,Razieh Abedi RadMohammad Reza YamaghaniAzamoss...,2025,https://link.springer.com/article/10.1007/s410...,Article,../1 - scrapping//result/springer_ijdsr_search...
2,Web-based phishing URL detection model using d...,International Journal of Data Science and Anal...,NaN,NaN,NaN,10.1007/s41060-025-00728-9,Kousik BarikSanjay MisraRaghini Mohan,2025,https://link.springer.com/article/10.1007/s410...,Article,../1 - scrapping//result/springer_ijdsr_search...
3,Node embedding approach for accurate detection...,International Journal of Data Science and Anal...,NaN,NaN,NaN,10.1007/s41060-024-00565-2,Nazar ZakiAnusuya KrishnanSherzod TuraevZahiri...,2024,https://link.springer.com/article/10.1007/s410...,Article,../1 - scrapping//result/springer_ijdsr_search...
4,A Survey on the Impact of Pre-Trained Language...,International Journal of Data Science and Anal...,NaN,NaN,NaN,10.1007/s41060-025-00805-z,Himanshu GautamAbhishek GaurDharmendra Kumar Y...,2025,https://link.springer.com/article/10.1007/s410...,Article,../1 - scrapping//result/springer_ijdsr_search...


In [87]:
# Web of Science
wos_files = [file for file in Path(f"{BASE_PATH}/result/").glob("WOS_*.xls")]

df_wos = read_list_of_excels(wos_files)
df_wos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1332 entries, 0 to 1331
Data columns (total 73 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Publication Type            1332 non-null   object 
 1   Authors                     1332 non-null   object 
 2   Book Authors                3 non-null      object 
 3   Book Editors                145 non-null    object 
 4   Book Group Authors          139 non-null    object 
 5   Author Full Names           1332 non-null   object 
 6   Book Author Full Names      3 non-null      object 
 7   Group Authors               7 non-null      object 
 8   Article Title               1332 non-null   object 
 9   Source Title                1332 non-null   object 
 10  Book Series Title           151 non-null    object 
 11  Book Series Subtitle        0 non-null      float64
 12  Language                    1332 non-null   object 
 13  Document Type               1332 

In [88]:
# IEEE Explore

ieee_files = [file for file in Path(f"{BASE_PATH}/result/").glob("IEEE_*.csv")]

df_ieee_explore = read_list_of_csvs(ieee_files)
df_ieee_explore.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1522 entries, 0 to 1521
Data columns (total 29 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Document Title          1522 non-null   object 
 1   Authors                 1522 non-null   object 
 2   Author Affiliations     1519 non-null   object 
 3   Publication Title       1522 non-null   object 
 4   Date Added To Xplore    1506 non-null   object 
 5   Publication Year        1522 non-null   int64  
 6   Volume                  214 non-null    object 
 7   Issue                   78 non-null     object 
 8   Start Page              1517 non-null   float64
 9   End Page                1517 non-null   object 
 10  Abstract                1522 non-null   object 
 11  ISSN                    568 non-null    object 
 12  ISBNs                   1350 non-null   object 
 13  DOI                     1511 non-null   object 
 14  Funding Information     165 non-null    

In [6]:
pd.set_option('display.max_rows', 500)

## Google Scholar

In [7]:

df_google_scholar.head()

# Inclusion Criterias
# 1. Includes Feature extraction
#   I'll read every single title to filter those that includes feature extraction
#   Afterwards, I'll read the abstracts of the remaining papers to further filter those that includes feature extraction
# 2. Text as regression model variables
#   I'll read every single title to filter those that includes text as regression model variables
#   Afterwards, I'll read the abstracts of the remaining papers to further filter those that includes text as regression model variables
# 3. Journal or Conference
#   I can filter using the str_article tags, such as "[HTML]" 
# 4. Published on the last 3 years (2025, 2024, 2023)
#  I'll filter using the 'Year' column


,str_search_string,nr_page,nr_order,str_article,nr_year,nr_citations,str_original_Link,str_pdf_link,category,filename
0,"""text regression"" AND (""feature extraction"" OR...",1,1,"Text regressionanalysis: A review, empirical, ...",2024.0,3,https://ieeexplore.ieee.org/abstract/document/...,https://ieeexplore.ieee.org/iel8/6287639/10380...,General,../1 - scrapping//result/google_scholar_search...
1,"""text regression"" AND (""feature extraction"" OR...",1,2,Multi-representation approach totext regressio...,2015.0,8,https://ieeexplore.ieee.org/abstract/document/...,https://www.fruct.org/files/publications/volum...,General,../1 - scrapping//result/google_scholar_search...
2,"""text regression"" AND (""feature extraction"" OR...",1,3,[HTML]TextRegress: A Python package for advanc...,2025.0,0,https://www.sciencedirect.com/science/article/...,https://www.sciencedirect.com/science/article/...,General,../1 - scrapping//result/google_scholar_search...
3,"""text regression"" AND (""feature extraction"" OR...",1,4,Gaussian processes fortext regression,2017.0,5,https://etheses.whiterose.ac.uk/id/eprint/17619/,https://etheses.whiterose.ac.uk/id/eprint/1761...,General,../1 - scrapping//result/google_scholar_search...
4,"""text regression"" AND (""feature extraction"" OR...",1,5,Analyzing online review helpfulness using a re...,2012.0,52,https://dl.acm.org/doi/abs/10.1145/2229156.222...,https://dl.acm.org/doi/pdf/10.1145/2229156.222...,General,../1 - scrapping//result/google_scholar_search...


In [8]:
# 4. Published on the last 3 years (2025, 2024, 2023)
print(df_google_scholar['nr_year'].value_counts())

total_before_filter = len(df_google_scholar)
df_google_scholar = df_google_scholar[df_google_scholar['nr_year'] >= 2023].copy()

print(f"Total articles before year filter: {total_before_filter}")
print(f"Total articles after year filter: {len(df_google_scholar)}")


nr_year
2022.0    64
2024.0    59
2025.0    58
2023.0    52
2021.0    44
2020.0    32
2019.0    23
2018.0    20
2017.0    19
2016.0    12
2015.0     7
2013.0     5
2014.0     1
2012.0     1
2008.0     1
2026.0     1
2011.0     1
2009.0     1
1992.0     1
2010.0     1
Name: count, dtype: int64
Total articles before year filter: 410
Total articles after year filter: 170


In [9]:
# 3. Journal or Conference
# As I can't say the article was reviewed by peers, let's only consider articles with at least one citation and with a valid link

total_before_filter = len(df_google_scholar)
df_google_scholar = df_google_scholar[(df_google_scholar['nr_citations'] > 0) & (df_google_scholar['str_original_Link'].notnull())].copy()

print(f"Total articles before journal/conference filter: {total_before_filter}")
print(f"Total articles after journal/conference filter: {len(df_google_scholar)}")


Total articles before journal/conference filter: 170
Total articles after journal/conference filter: 119


## International Journal of Data Science Review

In [10]:
# 4. Published on the last 3 years (2025, 2024, 2023)
print(df_springer_ijdsr['Publication Year'].value_counts())

total_before_filter = len(df_springer_ijdsr)
df_springer_ijdsr = df_springer_ijdsr[df_springer_ijdsr['Publication Year'] >= 2023].copy()

print(f"Total articles before year filter: {total_before_filter}")
print(f"Total articles after year filter: {len(df_springer_ijdsr)}")


Publication Year
2025    87
2024    57
2022    26
2023    26
2018     9
2021     7
2019     4
2017     4
2016     2
2026     2
2020     1
Name: count, dtype: int64
Total articles before year filter: 225
Total articles after year filter: 172


# Web Of Science

In [89]:


df_wos = df_wos[(df_wos['Publication Year'] >= 2023)].copy().reset_index(drop=True)


In [90]:

wos_screening = df_wos.apply(lambda row: evaluate_paper(
    title=row['Article Title'],
    abstract=row['Abstract'],
    inclusion_criteria=inclusion,
    exclusion_criteria=exclusion
), axis=1)


In [ ]:


df_wos_screening = pd.DataFrame(wos_screening.to_list())

df_wos_screening['Prisma Flow'] = (
    (df_wos_screening['IC I'] == True) & 
    (df_wos_screening['IC II'] == True) & 
    (df_wos_screening['EC I'] == False) & 
    (df_wos_screening['EC II'] == False) & 
    (df_wos_screening['EC III'] == False) & 
    (df_wos_screening['EC V'] == False)
)

df_wos_screening['Prisma Flow'].value_counts()


Prisma Flow
True     343
False    160
Name: count, dtype: int64

In [99]:

def get_step(row):
    if row['IC I'] == False:
        return 'IC III'
    elif row['IC II'] == False:
        return 'IC IV'
    elif row['EC I'] == True:
        return 'EC I'
    elif row['EC II'] == True:
        return 'EC II'
    elif row['EC III'] == True:
        return 'EC III'
    elif row['EC V'] == True:
        return 'EC V'
    else:
        return 'Included'

df_wos_screening['Step'] = df_wos_screening.apply(get_step, axis=1)
df_wos_screening.value_counts('Step')


Step
Included    343
IC III      130
EC V         25
IC IV         5
Name: count, dtype: int64

In [117]:


df_wos['IC III'] = df_wos_screening['IC I']
df_wos['IC IV'] = df_wos_screening['IC II']
df_wos['EC I'] = df_wos_screening['EC I']
df_wos['EC II'] = df_wos_screening['EC II']
df_wos['EC III'] = df_wos_screening['EC III']
df_wos['EC V'] = df_wos_screening['EC V']
df_wos['Step'] = df_wos_screening['Step']



In [118]:
import shutil

df_wos.to_excel(f'{THIS_FOLDER}/wos.xlsx', index=False)

shutil.copy2(
    f'{THIS_FOLDER}/wos.xlsx', 
    f'{MANUAL_FILTERING_PATH}/wos.xlsx'
)


'../3 - manual_filtering//wos.xlsx'

# IEEE Explore

In [94]:

df_ieee_explore = df_ieee_explore[df_ieee_explore['Publication Year'] >= 2023].copy().reset_index(drop=True)

In [112]:

ieee_screening = df_ieee_explore.apply(lambda row: evaluate_paper(
    title=row['Document Title'],
    abstract=row['Abstract'],
    inclusion_criteria=inclusion,
    exclusion_criteria=exclusion
), axis=1)


In [113]:

df_ieee_screening = pd.DataFrame(ieee_screening.to_list())

df_ieee_screening['Prisma Flow'] = (
    (df_ieee_screening['IC I'] == True) & 
    (df_ieee_screening['IC II'] == True) & 
    (df_ieee_screening['EC I'] == False) & 
    (df_ieee_screening['EC II'] == False) & 
    (df_ieee_screening['EC III'] == False) & 
    (df_ieee_screening['EC V'] == False)
)

df_ieee_screening['Prisma Flow'].value_counts()


Prisma Flow
True     606
False    182
Name: count, dtype: int64

In [114]:



def get_step(row):
    if row['IC I'] == False:
        return 'IC III'
    elif row['IC II'] == False:
        return 'IC IV'
    elif row['EC I'] == True:
        return 'EC I'
    elif row['EC II'] == True:
        return 'EC II'
    elif row['EC III'] == True:
        return 'EC III'
    elif row['EC V'] == True:
        return 'EC V'
    else:
        return 'Included'

df_ieee_screening['Step'] = df_ieee_screening.apply(get_step, axis=1)
df_ieee_screening.value_counts('Step')


Step
Included    606
EC V         99
IC III       44
IC IV        39
Name: count, dtype: int64

In [115]:

df_ieee_explore['IC III'] = df_ieee_screening['IC I']
df_ieee_explore['IC IV'] = df_ieee_screening['IC II']
df_ieee_explore['EC I'] = df_ieee_screening['EC I']
df_ieee_explore['EC II'] = df_ieee_screening['EC II']
df_ieee_explore['EC III'] = df_ieee_screening['EC III']
df_ieee_explore['EC V'] = df_ieee_screening['EC V']
df_ieee_explore['Step'] = df_ieee_screening['Step']


In [116]:
import shutil

df_ieee_explore.to_excel(f'{THIS_FOLDER}/ieee_explore.xlsx', index=False)

shutil.copy2(
    f'{THIS_FOLDER}/ieee_explore.xlsx', 
    f'{MANUAL_FILTERING_PATH}/ieee_explore.xlsx'
)


'../3 - manual_filtering//ieee_explore.xlsx'

# Standardize to export

In [12]:

desired_format = {
    'title': str,
    'year': int, 
    'citations': int,
    'link': str,
    'pdf_link': str,
    'category': str,
    'source': str
}


In [13]:

df_rows = []

for _, row in df_google_scholar.iterrows():
    df_row = {
        'title': row['str_article'],
        'year': row['nr_year'],
        'citations': row['nr_citations'],
        'link': row['str_original_Link'],
        'pdf_link': row['str_pdf_link'],
        'category': row['category'],
        'source': 'Google Scholar'
    }
    df_rows.append(df_row)

for _, row in df_springer_ijdsr.iterrows():
    df_row = {
        'title': row['Item Title'],
        'year': row['Publication Year'],
        'citations': -1,
        'link': row['URL'],
        'pdf_link': None,
        'category': row['filename'].split('_')[-1].replace('.csv', ''),
        'source': 'Springer International Journal of Data Science and Analytics'
    }
    df_rows.append(df_row)

df_combined = pd.DataFrame(df_rows)
df_combined = df_combined.astype(desired_format)

print(df_combined.shape)

df_combined.head()

(291, 7)


,title,year,citations,link,pdf_link,category,source
0,"Text regressionanalysis: A review, empirical, ...",2024,3,https://ieeexplore.ieee.org/abstract/document/...,https://ieeexplore.ieee.org/iel8/6287639/10380...,General,Google Scholar
1,Dating Greek papyri withtext regression,2023,8,https://aclanthology.org/2023.acl-long.556/,https://aclanthology.org/2023.acl-long.556.pdf,General,Google Scholar
2,[HTML]Residential load forecasting based on lo...,2024,8,https://www.mdpi.com/2071-1050/16/24/11252,https://www.mdpi.com/2071-1050/16/24/11252,General,Google Scholar
3,Regression applied to legal judgments to predi...,2023,7,https://peerj.com/articles/cs-1225/,https://peerj.com/articles/cs-1225.pdf,General,Google Scholar
4,Forecasting Scientific Impact: A Model for Pre...,2025,1,http://iapress.org/index.php/soic/article/view...,http://iapress.org/index.php/soic/article/down...,General,Google Scholar


In [14]:

# Group same article that appears in multiple categories
df_combined_uniques = (

    df_combined
        .groupby([name for name in df_combined.columns if name != 'category'])
        .agg(list)
        .reset_index()

)

print(df_combined_uniques.shape)
display(df_combined_uniques.head())

(200, 7)


,title,year,citations,link,pdf_link,source,category
0,<i>SeNSe</i>: embedding alignment via semantic...,2024,-1,https://link.springer.com/article/10.1007/s410...,None,Springer International Journal of Data Science...,"[Dense Embeddings, Applied cases]"
1,A MultilingualBERT EmbeddingsApproach in Ident...,2024,2,https://ieeexplore.ieee.org/abstract/document/...,nan,Google Scholar,[Contextual Embeddings]
2,A Pretraining Approach for Small-sample Traini...,2025,1,https://link.springer.com/article/10.1007/s109...,https://linchin.ndmctsgh.edu.tw/papers/2025/20...,Google Scholar,[Contextual Embeddings]
3,A Survey on the Impact of Pre-Trained Language...,2025,-1,https://link.springer.com/article/10.1007/s410...,None,Springer International Journal of Data Science...,"[Classic Methods, Dense Embeddings, Comparison..."
4,A common-specific feature cross-fusion attenti...,2024,-1,https://link.springer.com/article/10.1007/s410...,None,Springer International Journal of Data Science...,"[Dense Embeddings, Comparison of methods]"


## Export

In [15]:
import shutil

df_combined_uniques.to_excel(f'{THIS_FOLDER}/combined_unique_articles.xlsx', index=False)

shutil.copy2(
    f'{THIS_FOLDER}/combined_unique_articles.xlsx', 
    f'{MANUAL_FILTERING_PATH}/combined_unique_articles.xlsx'
)


'../3 - manual_filtering//combined_unique_articles.xlsx'